## Importing Libraries

In [1]:
import os
import time
os.makedirs("../data/processed", exist_ok=True)

import pandas as pd

import sys
sys.path.append("../")  
from src.data_collection import enrich_film, get_movie_details, get_movie_keywords
# in order to use these modules, I'm using an API_KEY. If you try to rerun the code, it'll fail if you don't add your API_KEY

## Loading Datasets

We have several datasets, we'll load all of them here. We'll see what each one of them have, 

### MovieLens

MovieLens will be our base. There are two reasons for this:
- It has the links dataset which will be the core of our future database in SQL
- They are the cleanest and clearest datasets
- It only has movies. IMBD includes some TVshows and other stuff, not only movies.

It has almost 90.000 records which will make also all our datasets more manageable

In [2]:
links = pd.read_csv("../data/raw/movielens/links.csv")
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [3]:
links.shape

(86537, 3)

In [4]:
movies = pd.read_csv("../data/raw/movielens/movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
movies.shape

(86537, 3)

In [6]:
ratings = pd.read_csv("../data/raw/movielens/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


In [7]:
ratings.shape

(33832162, 4)

I'm loading now some datasets that may not be useful for analysis, but could help a lot with ML. 

In [10]:
genome_tags = pd.read_csv("../data/raw/movielens/genome-tags.csv")
genome_tags.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [11]:
genome_tags.shape

(1128, 2)

In [12]:
genome_scores = pd.read_csv("../data/raw/movielens/genome-scores.csv")
genome_scores.head()

,movieId,tagId,relevance
0,1,1,0.03200
1,1,2,0.02225
2,1,3,0.07000
3,1,4,0.05900
4,1,5,0.12300


In [13]:
genome_scores.shape

(18472128, 3)

### IMDB

These datasets have a lot of information and require a lot of cleaning and decision making on several areas. However, we'll do that on the cleaning part. We'll just load them here.

In [14]:
title_basics = pd.read_csv("../data/raw/imbd/title.basics.tsv", sep='\t')
title_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [15]:
title_basics.shape

(12508399, 9)

In [16]:
title_ratings = pd.read_csv("../data/raw/imbd/title.ratings.tsv", sep='\t')
title_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2216
1,tt0000002,5.5,318
2,tt0000003,6.4,2334
3,tt0000004,5.1,199
4,tt0000005,6.2,3052


In [17]:
title_ratings.shape

(1670967, 3)

### Kaggle

The original idea was to use the Bechdel API, but it had been discontinued. 

So, even though, this is a clear limitation it is still a good option in order to include a gender perspective. 

In [18]:
bechdel = pd.read_csv("../data/external/bechdel_detailed.csv", index_col = 'Unnamed: 0')
bechdel.head()

,title,year,rating,dubious,imdbid,id,submitterid,date,visible
0,Passage de Venus,1874.0,0.0,0.0,3155794.0,9602.0,18880.0,2021-04-02 20:58:09,1.0
1,La Rosace Magique,1877.0,0.0,0.0,14495706.0,9804.0,19145.0,2021-05-11 00:11:22,1.0
2,Sallie Gardner at a Gallop,1878.0,0.0,0.0,2221420.0,9603.0,18882.0,2021-04-03 02:25:27,1.0
3,Le singe musicien,1878.0,0.0,0.0,12592084.0,9806.0,19151.0,2021-05-11 23:38:54,1.0
4,Athlete Swinging a Pick,1881.0,0.0,0.0,7816420.0,9816.0,19162.0,2021-05-13 01:32:14,1.0


In [19]:
bechdel.shape

(9373, 9)

### TMDB API

The Movie Database (TMDB) API will help us get information from the budget and revenue to film's posters for our Streamlit app.

In [20]:
# Test with The Godfather (tmdb_id = 238)
movie = get_movie_details(238)
keywords = get_movie_keywords(238)

print(f"Title: {movie['title']}")
print(f"Budget: ${movie['budget']:,}")
print(f"Revenue: ${movie['revenue']:,}")
print(f"Keywords: {[k['name'] for k in keywords['keywords'][:10]]}")

Title: The Godfather
Budget: $6,000,000
Revenue: $245,066,411
Keywords: ['based on novel or book', 'loss of loved one', 'love at first sight', 'italy', 'gangster', 'symbolism', 'patriarch', 'europe', 'organized crime', 'mafia']


## Cleaning the Datasets

### Formatting columns names

First thing I want to do is standarizing column names of every dataset, so they all have the same format. 

In [ ]:
datasets = [links, movies, ratings, genome_tags, genome_scores,
            title_basics, title_ratings,
            bechdel]
for df in datasets:
    print(df.columns)

Index(['movieId', 'imdbId', 'tmdbId'], dtype='object')
Index(['movieId', 'title', 'genres'], dtype='object')
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')
Index(['userId', 'movieId', 'tag', 'timestamp'], dtype='object')
Index(['tagId', 'tag'], dtype='object')
Index(['movieId', 'tagId', 'relevance'], dtype='object')
Index(['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult',
       'startYear', 'endYear', 'runtimeMinutes', 'genres'],
      dtype='object')
Index(['tconst', 'averageRating', 'numVotes'], dtype='object')
Index(['title', 'year', 'rating', 'dubious', 'imdbid', 'id', 'submitterid',
       'date', 'visible'],
      dtype='object')


In [22]:
col_map = {
    'movieId' : 'movie_id',
    'imdbId' : 'imdb_id',
    'tmdbId' : 'tmdb_id',
    'title' : 'title',
    'genres' : 'genres',
    'userId' : 'user_id',
    'rating' : 'rating',
    'timestamp' : 'duration',
    'tag' : 'tag',
    'tagId' : 'tag_id',
    'relevance' : 'relevance',
    'nconst' : 'name_id',
    'primaryName' : 'name',
    'birthYear' : 'birth_year',
    'deathYear' : 'death_year',
    'primaryProfession' : 'profession',
    'knownForTitles' : 'known_for',
    'tconst' : 'imdb_id',
    'titleType' : 'title_type',
    'primaryTitle' : 'title',
    'originalTitle' : 'original_title',
    'isAdult' : 'is_adult',
    'startYear' : 'year',
    'endYear' : 'end_year',
    'runtimeMinutes' : 'runtime_min',
    'directors' : 'directors',
    'writers' : 'writers',
    'ordering' : 'row_per_title',
    'category' : 'category',
    'job' : 'job',
    'characters' : 'characters',
    'averageRating' : 'avg_rating',
    'numVotes' : 'num_votes',
    'year' : 'year',
    'dubious' : 'dubious',
    'imdbid' : 'imdb_id',
    'id' : 'bechdel_id',
    'submitterid' : 'user_id',
    'date' : 'submit_date',
    'visible' : 'visible',
}

# some keys of the dictionaries are from past .csv files that have been dropped of the project in the end. 
# leaving them just in case

def rename_columns(df, col_map = col_map):
    df.rename(mapper = col_map, axis = 1, inplace = True)
    return df

for df in datasets:
    rename_columns(df)

In [23]:
links.head()

#it worked :))

,movie_id,imdb_id,tmdb_id
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


### Individual cleaning of datasets

- We'll see what columns are useful, what to keep and what to drop.
- We'll check datatypes and change them when needed
- We'll format null values as now there are null values that pandas doesn't detect. 
- We'll decide what to do with them. 
- We'll handle duplicates as well.

#### MovieLens datasets

##### Links dataset

In [24]:
links.head()

,movie_id,imdb_id,tmdb_id
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [25]:
links.dtypes

movie_id      int64
imdb_id       int64
tmdb_id     float64
dtype: object

In [26]:
links["imdb_id"] = "tt" + links["imdb_id"].astype(str).str.zfill(7)
# we transform the imdb_id so it has the same formatting of the imdb_ids in imdb datasets

links.head()

,movie_id,imdb_id,tmdb_id
0,1,tt0114709,862.0
1,2,tt0113497,8844.0
2,3,tt0113228,15602.0
3,4,tt0114885,31357.0
4,5,tt0113041,11862.0


In [27]:
links["tmdb_id"] = links["tmdb_id"].astype("Int64") 
links.head()


,movie_id,imdb_id,tmdb_id
0,1,tt0114709,862
1,2,tt0113497,8844
2,3,tt0113228,15602
3,4,tt0114885,31357
4,5,tt0113041,11862


In [28]:
links.dtypes

movie_id     int64
imdb_id     object
tmdb_id      Int64
dtype: object

In [29]:
links.isna().sum()

movie_id      0
imdb_id       0
tmdb_id     126
dtype: int64

In [30]:
links[links["tmdb_id"].isna()]

,movie_id,imdb_id,tmdb_id
706,721,tt0114103,<NA>
715,730,tt0125877,<NA>
754,770,tt0038426,<NA>
775,791,tt0113610,<NA>
1080,1107,tt0102336,<NA>
...,...,...,...
69061,222211,tt7688638,<NA>
69755,224398,tt5868094,<NA>
72468,234301,tt5714216,<NA>
73108,237231,tt9372822,<NA>


We'll drop the null values. They are a small sample and we'll make our final data much cleaner.

In [31]:
links = links.dropna()

##### Movies dataset

In [32]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [33]:
movies.dtypes

movie_id     int64
title       object
genres      object
dtype: object

In [34]:
# extracts year
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')

# removes year from title
movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

movies.head()

,movie_id,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [35]:
movies.isna().sum()

movie_id      0
title         0
genres        0
year        618
dtype: int64

In [36]:
movies[movies["year"].isna()]

,movie_id,title,genres,year
15038,79607,"Millions Game, The (Das Millionenspiel)",Action|Drama|Sci-Fi|Thriller,NaN
25443,123619,Terrible Joe Moran,(no genres listed),NaN
26374,125571,The Court-Martial of Jackie Robinson,(no genres listed),NaN
26399,125632,In Our Garden,(no genres listed),NaN
26484,125958,Stephen Fry In America - New World,(no genres listed),NaN
...,...,...,...,...
86282,288069,American Masters: Miles Davis - Birth of the Cool,Documentary,NaN
86365,288447,A House A Little Too Quiet,(no genres listed),NaN
86432,288661,Sabotage,(no genres listed),NaN
86455,288725,We Have Never Been Modern,Crime,NaN


In [37]:
movies["genres"].value_counts()

genres
Drama                                             12246
Documentary                                        8064
Comedy                                             7689
(no genres listed)                                 7060
Comedy|Drama                                       3196
                                                  ...  
Action|Adventure|Crime|Drama|Sci-Fi                   1
Adventure|Animation|Children|Comedy|Drama             1
Action|Adventure|Crime|IMAX                           1
Action|Adventure|Comedy|Crime|Mystery|Thriller        1
Action|Adventure|Drama|Horror|Mystery|Thriller        1
Name: count, Length: 1796, dtype: int64

In [38]:
movies['genres'] = movies['genres'].replace("(no genres listed)", pd.NA)

In [39]:
movies['genres'].isna().sum()

# as they are having lots of null values. we are dropping this column entirely. we'll use the IMDB genres as they are more complete. 

np.int64(7060)

In [40]:
movies = movies.drop(columns = ["genres"])
movies = movies.drop(columns = ["year"]) # as before, they have more null values than IMDB ones
movies = movies.drop(columns = ["title"]) # as we are relying more on IMDB datasets, we'll keep those titles. this dataset won't be used in the end.

In [41]:
movies.isna().sum()

movie_id    0
dtype: int64

##### Ratings dataset

In [42]:
ratings.head()

,user_id,movie_id,rating,duration
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


In [43]:
ratings.dtypes

user_id       int64
movie_id      int64
rating      float64
duration      int64
dtype: object

In [44]:
ratings.isna().sum()

user_id     0
movie_id    0
rating      0
duration    0
dtype: int64

In [45]:
ratings.duplicated().sum()

np.int64(0)

In [46]:
# in the metadata I found that "Timestamps represent seconds since midnight Coordinated Universal Time (UTC) of January 1, 1970."
# we are dropping that column

ratings.drop(columns = ["duration"], inplace = True)
ratings.head()


,user_id,movie_id,rating
0,1,1,4.0
1,1,110,4.0
2,1,158,4.0
3,1,260,4.5
4,1,356,5.0


##### Genome Tags dataset

In [52]:
genome_tags.head()

,tag_id,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [53]:
genome_tags.dtypes

tag_id     int64
tag       object
dtype: object

In [54]:
genome_tags.isna().sum()

tag_id    0
tag       0
dtype: int64

In [55]:
genome_tags.duplicated().sum()

np.int64(0)

##### Genome Score dataset

In [56]:
genome_scores.head()

,movie_id,tag_id,relevance
0,1,1,0.03200
1,1,2,0.02225
2,1,3,0.07000
3,1,4,0.05900
4,1,5,0.12300


In [57]:
genome_scores.dtypes

movie_id       int64
tag_id         int64
relevance    float64
dtype: object

In [58]:
genome_scores.isna().sum()

movie_id     0
tag_id       0
relevance    0
dtype: int64

In [59]:
genome_scores.duplicated().sum()

np.int64(0)

In [60]:
genome_scores[genome_scores["relevance"] > 1]

,movie_id,tag_id,relevance


#### IMDB Datasets

The most important thing with the IMDB Datasets is keeping just the films that are already inside the MovieLens dataset. In order to do that, we are running the following code:

In [61]:
title_basics = title_basics[
    (title_basics['imdb_id'].isin(links["imdb_id"])) &  # only MovieLens films
    (title_basics['title_type'] == 'movie') &          # only films
    (title_basics['is_adult'] == 0)                  # no adult content
]

# we have filtered the one containing mostly all the info about films. we'll start cleaning that one. 
# later, we'll do the same filtering for the rest of the datasets.

We are going to filter all the title datasets that are the ones containing the films so that every dataset contain the required films. 

In all the IMDB datasets null values won't be detected by pandas as they are a string, so, we'll change that as well.  Also, genres and list of things are separated by commas, we'll standarize that using | as it was in the MovieLens one, but this will be individually.

In [62]:
def same_films(df, mask = title_basics):
    new_df = df[df["imdb_id"].isin(mask["imdb_id"])]
    return new_df

title_basics = same_films(title_basics)
title_ratings = same_films(title_ratings)

imdb_data = [title_basics, title_ratings]

def null_values(df):
    new_df = df.replace(to_replace="\\N", value=pd.NA)
    return new_df
    
title_basics = null_values(title_basics)
title_ratings = null_values(title_ratings)

##### Title Basics dataset

In [63]:
title_basics.head()

,imdb_id,title_type,title,original_title,is_adult,year,end_year,runtime_min,genres
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,<NA>,70,"Action,Adventure,Biography"
2076,tt0002101,movie,Cleopatra,Cleopatra,0,1912,<NA>,100,"Drama,History"
2105,tt0002130,movie,Dante's Inferno,L'inferno,0,1911,<NA>,71,"Adventure,Drama,Fantasy"
2396,tt0002423,movie,Passion,Madame DuBarry,0,1919,<NA>,113,"Biography,Drama,Romance"
2418,tt0002445,movie,Quo Vadis?,Quo Vadis?,0,1913,<NA>,120,"Drama,History"


In [64]:
title_basics["genres"] = title_basics["genres"].str.replace(",", "|")
title_basics.head()

,imdb_id,title_type,title,original_title,is_adult,year,end_year,runtime_min,genres
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,<NA>,70,Action|Adventure|Biography
2076,tt0002101,movie,Cleopatra,Cleopatra,0,1912,<NA>,100,Drama|History
2105,tt0002130,movie,Dante's Inferno,L'inferno,0,1911,<NA>,71,Adventure|Drama|Fantasy
2396,tt0002423,movie,Passion,Madame DuBarry,0,1919,<NA>,113,Biography|Drama|Romance
2418,tt0002445,movie,Quo Vadis?,Quo Vadis?,0,1913,<NA>,120,Drama|History


In [65]:
title_basics.dtypes

imdb_id           object
title_type        object
title             object
original_title    object
is_adult           int64
year              object
end_year          object
runtime_min       object
genres            object
dtype: object

In [66]:
title_basics.isna().sum()

# the start year may be completed with the movielens dataset
# we'll keep the null values in the runtime as well. 
# 70 genres won't be that important with the amount of data we have already. I prefer to keep them anyway.

imdb_id               0
title_type            0
title                 0
original_title        0
is_adult              0
year                 12
end_year          70475
runtime_min         185
genres               69
dtype: int64

In [67]:
title_basics.drop(columns = ["end_year"], inplace = True) # this column only makes sense for tvshows
title_basics.drop(columns = ["original_title"], inplace = True) # I prefer to keep the famous name only
title_basics.drop(columns = ["title_type"], inplace = True) # Once we have filter by movies it makes no sense to keep this
title_basics.drop(columns = ["is_adult"], inplace = True) # Once we have filter this it makes no sense to keep this

In [68]:
title_basics[["year", "runtime_min"]] = title_basics[["year", "runtime_min"]].astype("Int64")

In [69]:
title_basics.head()

,imdb_id,title,year,runtime_min,genres
570,tt0000574,The Story of the Kelly Gang,1906,70,Action|Adventure|Biography
2076,tt0002101,Cleopatra,1912,100,Drama|History
2105,tt0002130,Dante's Inferno,1911,71,Adventure|Drama|Fantasy
2396,tt0002423,Passion,1919,113,Biography|Drama|Romance
2418,tt0002445,Quo Vadis?,1913,120,Drama|History


We are creating a new dataset for genres as we'll need them separated, so that we can actually count categories separately. 

In [70]:
title_genres = (
    title_basics[['imdb_id', 'genres']]
    .assign(genre = title_basics['genres'].str.split('|'))
    .explode('genre')
    .reset_index(drop=True)
)
title_genres.head()

,imdb_id,genres,genre
0,tt0000574,Action|Adventure|Biography,Action
1,tt0000574,Action|Adventure|Biography,Adventure
2,tt0000574,Action|Adventure|Biography,Biography
3,tt0002101,Drama|History,Drama
4,tt0002101,Drama|History,History


In [71]:
title_genres = title_genres.drop(columns = ["genres"])
title_basics = title_basics.drop(columns = ["genres"])

##### Title Ratings dataset

In [72]:
title_ratings.head()

,imdb_id,avg_rating,num_votes
545,tt0000574,6.0,1069
1269,tt0002101,5.1,675
1277,tt0002130,7.1,4125
1385,tt0002423,6.7,1130
1393,tt0002445,6.1,536


In [73]:
title_ratings.dtypes

imdb_id        object
avg_rating    float64
num_votes       int64
dtype: object

In [74]:
title_ratings.isna().sum()

imdb_id       0
avg_rating    0
num_votes     0
dtype: int64

#### Bechdel dataset

In [75]:
bechdel.head()

,title,year,rating,dubious,imdb_id,bechdel_id,user_id,submit_date,visible
0,Passage de Venus,1874.0,0.0,0.0,3155794.0,9602.0,18880.0,2021-04-02 20:58:09,1.0
1,La Rosace Magique,1877.0,0.0,0.0,14495706.0,9804.0,19145.0,2021-05-11 00:11:22,1.0
2,Sallie Gardner at a Gallop,1878.0,0.0,0.0,2221420.0,9603.0,18882.0,2021-04-03 02:25:27,1.0
3,Le singe musicien,1878.0,0.0,0.0,12592084.0,9806.0,19151.0,2021-05-11 23:38:54,1.0
4,Athlete Swinging a Pick,1881.0,0.0,0.0,7816420.0,9816.0,19162.0,2021-05-13 01:32:14,1.0


In [76]:
bechdel['imdb_id'] = bechdel['imdb_id'].astype('Int64').astype(str)
bechdel["imdb_id"] = "tt" + bechdel["imdb_id"].astype(str).str.zfill(7)
bechdel.head()

,title,year,rating,dubious,imdb_id,bechdel_id,user_id,submit_date,visible
0,Passage de Venus,1874.0,0.0,0.0,tt3155794,9602.0,18880.0,2021-04-02 20:58:09,1.0
1,La Rosace Magique,1877.0,0.0,0.0,tt14495706,9804.0,19145.0,2021-05-11 00:11:22,1.0
2,Sallie Gardner at a Gallop,1878.0,0.0,0.0,tt2221420,9603.0,18882.0,2021-04-03 02:25:27,1.0
3,Le singe musicien,1878.0,0.0,0.0,tt12592084,9806.0,19151.0,2021-05-11 23:38:54,1.0
4,Athlete Swinging a Pick,1881.0,0.0,0.0,tt7816420,9816.0,19162.0,2021-05-13 01:32:14,1.0


In [77]:
bechdel = same_films(bechdel)
bechdel
# even though this is one of our most important datasets and it is also one of the smaller, we need to drop some rows as we need to have films that are already on our other two datasets.

,title,year,rating,dubious,imdb_id,bechdel_id,user_id,submit_date,visible
116,"Story of the Kelly Gang, The",1906.0,1.0,0.0,tt0000574,1349.0,1469.0,2010-07-26 21:01:14,1.0
132,Cleopatra,1912.0,2.0,0.0,tt0002101,2003.0,2676.0,2011-02-07 06:12:54,1.0
137,A Florida Enchantment,1914.0,2.0,1.0,tt0003973,4457.0,7991.0,2013-08-10 05:01:03,1.0
138,"Birth of a Nation, The",1915.0,2.0,0.0,tt0004972,1258.0,1359.0,2010-07-23 00:46:01,1.0
139,Gretchen the Greenhorn,1916.0,3.0,0.0,tt0006745,2008.0,2685.0,2011-02-08 06:49:30,1.0
...,...,...,...,...,...,...,...,...,...
9367,Encanto,2021.0,3.0,0.0,tt2953050,10151.0,19732.0,2021-12-02 00:36:48,1.0
9368,Love Hard,2021.0,2.0,0.0,tt10752004,10152.0,19735.0,2021-12-05 19:22:20,1.0
9369,Cruella,2021.0,3.0,0.0,tt3228774,9861.0,19231.0,2021-06-01 03:16:58,1.0
9370,West Side Story,2021.0,3.0,0.0,tt3581652,10157.0,19743.0,2021-12-10 03:10:09,1.0


In [78]:
bechdel = bechdel.drop(columns = ["bechdel_id", "user_id", "submit_date"]) # we don't need more ids and we don't need the submit date either
bechdel = bechdel.drop(columns = ["title", "year"]) # we already have these data in other datasets
bechdel = bechdel.drop(columns = ["visible"]) # doesn't give information

In [79]:
bechdel = bechdel[["imdb_id", "rating", "dubious"]]
bechdel.head()

,imdb_id,rating,dubious
116,tt0000574,1.0,0.0
132,tt0002101,2.0,0.0
137,tt0003973,2.0,1.0
138,tt0004972,2.0,0.0
139,tt0006745,3.0,0.0


In [ ]:
bechdel = bechdel["imdb_id"].drop_duplicates()
bechdel = bechdel.drop_duplicates()

In [80]:
print(bechdel["rating"].value_counts())
print("\n")
print(bechdel["dubious"].value_counts())

rating
3.0    4652
1.0    1877
2.0     869
0.0     802
Name: count, dtype: int64


dubious
0.0    7179
1.0     734
Name: count, dtype: int64


In [81]:
bechdel[bechdel["dubious"] == 1]["rating"].value_counts()
# I was thinking on dropping dubious. I don't know if it'll be useful so I'll keep it, for the moment.

rating
3.0    545
2.0    102
1.0     67
0.0     20
Name: count, dtype: int64

#### MovieLens datasets

We did the filtering in the IMDB datasets with MovieLens as the base. However, we did use some filter after that (the is_adult and the title_type), so there is still a difference in rows and films between datasets. 

In [82]:
links = same_films(links)
links.head()

,movie_id,imdb_id,tmdb_id
0,1,tt0114709,862
1,2,tt0113497,8844
2,3,tt0113228,15602
3,4,tt0114885,31357
4,5,tt0113041,11862


In [83]:
links.shape

(70475, 3)

Now, we'll do the same but with MovieLens datasets using movie_id. This is the last step in order to have the same films in every dataset.

In [ ]:
def same_movies_ids(df, mask = links):
    new_df = df[df["movie_id"].isin(mask["movie_id"])]
    return new_df


movies = same_movies_ids(movies)
ratings = same_movies_ids(ratings)
genome_scores = same_movies_ids(genome_scores)

# not doing genome_tags as it doesn't use movie_id

## Save copies of the Datasets

In [116]:
datasets_dict = {
    "links": links,
    "ratings": ratings,
    "genome_tags": genome_tags,
    "genome_scores": genome_scores,
    "title_basics": title_basics,
    "title_genres": title_genres,
    "title_ratings": title_ratings,
    "bechdel": bechdel
}

for data in datasets_dict.keys():
    datasets_dict[data].to_csv(f"../data/processed/{data}.csv")

## Creating the Dataframe with the API

In [90]:
cache_file = "../data/processed/tmdb_financial.csv"

# Resumes from cache if it exists
if os.path.exists(cache_file):
    cache = pd.read_csv(cache_file)
    already_done = set(cache['tmdb_id'].tolist())
    results = cache.to_dict('records')
    print(f"Resuming: {len(already_done)} films already done")
else:
    already_done = set()
    results = []

tmdb_ids = links['tmdb_id'].tolist()
remaining = [tid for tid in tmdb_ids if tid not in already_done]
print(f"Films to process: {len(remaining)}")

for i, tmdb_id in enumerate(remaining):
    try:
        result = enrich_film(tmdb_id)
        
        result['budget'] = result['budget'] if result['budget'] > 0 else None
        result['revenue'] = result['revenue'] if result['revenue'] > 0 else None
        
        results.append(result)
    except Exception as e:
        print(f"Error on {tmdb_id}: {e}")
        results.append({'tmdb_id': tmdb_id, 'budget': None, 'revenue': None})

    # Sleeps every 30 calls
    if (i + 1) % 30 == 0:
        time.sleep(1)

    # Checkpoint every 500 films
    if (i + 1) % 500 == 0:
        pd.DataFrame(results).to_csv(cache_file, index=False)
        print(f"{i + 1}/{len(remaining)} processed")

tmdb_enriched = pd.DataFrame(results)
tmdb_enriched.to_csv(cache_file)
print(f"Done — {len(tmdb_enriched)} films collected")

Resuming: 70469 films already done
Films to process: 0
Done — 70470 films collected


In [101]:
tmdb_enriched.head()

,tmdb_id,budget,revenue
0,862,30000000.0,401157969.0
1,8844,65000000.0,262821940.0
2,15602,25000000.0,71518503.0
3,31357,16000000.0,81452156.0
4,11862,NaN,76594107.0


In [ ]:
tmdb_enriched.isna().sum()

Unnamed: 0        0
tmdb_id           0
budget        55223
revenue       54732
dtype: int64

In [115]:
tmdb_enriched[~(tmdb_enriched["budget"].isna()) & ~(tmdb_enriched["revenue"].isna())].nunique()

tmdb_id    10390
budget      1479
revenue     8715
dtype: int64

We have the complete financial data for 10.000 films. It could be more, but it's not a bad amount.